<a href="https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
!pip install -q duckdb huggingface_hub pandas scikit-learn

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download, HfApi
from google.colab import userdata

token = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

dim_clients_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset",
    filename="dim_clients.parquet", token=token)
dim_content_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset",
    filename="dim_content.parquet", token=token)
fact_daily_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet", token=token)

con = duckdb.connect()
con.execute(f"CREATE VIEW dim_clients AS SELECT * FROM read_parquet('{dim_clients_path}')")
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")
con.execute(f"CREATE VIEW fact_daily AS SELECT * FROM read_parquet('{fact_daily_path}')")

print(con.execute("SELECT COUNT(*) FROM fact_daily").fetchone())
print(con.execute("SELECT MIN(report_date), MAX(report_date) FROM fact_daily").fetchone())

(9841378,)
(datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page's daily performance snapshot, for one client, on
one date — combining Search Console signals (impressions, clicks, average
position) and GA4 signals (sessions, engaged sessions, engagement time) for
that content_hash_id on that report_date.

Time window: March 2026 (2026-03-01 to 2026-03-31) — a mid-panel month with
9,841,378 rows. Not June 2026, which is the sealed final month.

In [16]:
grain_check = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM fact_daily
    GROUP BY 1,2,3
    ORDER BY row_count DESC
    LIMIT 5
""").fetchdf()
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count
0,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,1
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,1


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, ga4_sessions, ga4_engaged_sessions,
         ga4_total_engagement_sec

Label: whether gsc_avg_position improves on the following day for that
       content page (a proxy for ranking-signal movement)

Context: client_hash_id, content_hash_id, report_date, client_has_gsc,
         client_has_ga4, gsc_data_available, ga4_data_available — keys and
         availability flags, not predictive signal themselves

Excluded: gsc_avg_position as a feature — it's what the label is derived
          from, so including it would leak the outcome directly into the
          input. The ai_* referral columns (ai_chatgpt, ai_perplexity, etc.)
          are also excluded, to keep this lane focused on classic
          search-ranking signals rather than mixing in AI-referral traffic.

In [17]:
fields_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(gsc_impressions) AS has_impressions,
        COUNT(gsc_clicks) AS has_clicks,
        COUNT(ga4_sessions) AS has_sessions,
        COUNT(ga4_engaged_sessions) AS has_engaged_sessions,
        COUNT(ga4_total_engagement_sec) AS has_engagement_sec
    FROM fact_daily
""").fetchdf()
fields_check


,total_rows,has_impressions,has_clicks,has_sessions,has_engaged_sessions,has_engagement_sec
0,9841378,9841378,9841378,6822637,6822637,6822637


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Availability: filtering to rows where the client actually has GSC data
connected and available, using IS TRUE, and reporting how many rows survive.

In [18]:
availability_check = con.execute("""
    SELECT COUNT(*) AS available_rows
    FROM fact_daily
    WHERE client_has_gsc IS TRUE
      AND gsc_data_available IS TRUE
""").fetchdf()
availability_check


,available_rows
0,3611061


Missing values: checking how many rows have nulls in the core metrics even
after the availability filter.

In [19]:
missing_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position
    FROM fact_daily
    WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
""").fetchdf()
missing_check

,total_rows,missing_impressions,missing_clicks,missing_position
0,3611061,0.0,0.0,0.0


Window check: confirming the full row count and date span line up with the
March 2026 window stated in Section 1.

In [20]:
window_check = con.execute("""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM fact_daily
""").fetchdf()
window_check

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell you:

- Unbalanced history: not every client has the same amount of history in
  this window — some have data for all 31 days of March, others far fewer,
  so month-over-month comparisons rest on very different amounts of history
  per client.

- GSC-only early rows: some rows have client_has_gsc = TRUE but
  client_has_ga4 = FALSE (or ga4_data_available = FALSE), so engagement
  signals are absent — not zero, just missing — for those rows. Any join
  or feature mixing GSC and GA4 silently drops or biases toward the subset
  of clients with both connected.

- Window overlaps: this table is daily-grain, so it can't distinguish a
  genuine ranking shift on a given day from noise or reporting lag/smoothing
  that Search Console applies internally across nearby days — no
  window-boundary correction is visible at this grain.

In [21]:
limits_check = con.execute("""
    SELECT
        client_hash_id,
        COUNT(DISTINCT report_date) AS days_present,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_days,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_days
    FROM fact_daily
    GROUP BY client_hash_id
    ORDER BY days_present ASC
    LIMIT 10
""").fetchdf()
limits_check

,client_hash_id,days_present,gsc_available_days,ga4_available_days
0,client_e00b29e582949543,9,52.0,109.0
1,client_810019792c9b8efc,12,84.0,14.0
2,client_f6f0cdf26d03d7bd,13,91.0,0.0
3,client_86ebc2f12c01f586,29,1072.0,1066.0
4,client_d211cb07b9059bab,31,1387.0,796.0
5,client_a2eeb8899886adde,31,1477.0,343.0
6,client_c182d11e4862a37d,31,24095.0,912.0
7,client_62f4a7e64f5e0096,31,610971.0,0.0
8,client_fef1a8f436438636,31,254118.0,48001.0
9,client_8ae2bfb5aa1ffa1e,31,385.0,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.